# IA Responsable: Casos prácticos y mecanismos de control

Este notebook reúne y explica en detalle las preguntas trabajadas sobre **IA Responsable**, sesgos algorítmicos, uso seguro de LLMs, privacidad de datos con Amazon Bedrock y cumplimiento regulatorio en AWS.

## Objetivos

Al finalizar, deberías poder:

1. Identificar casos de sesgo algorítmico en modelos de decisión.
2. Relacionar problemas reales con los pilares de IA Responsable.
3. Diseñar mecanismos para reducir alucinaciones y contenido ofensivo en LLMs.
4. Proteger datos personales al usar servicios de IA como Amazon Bedrock.
5. Reconocer servicios y prácticas de AWS útiles para cumplimiento con GDPR e HIPAA.


---
# 1. Caso: modelo de IA para aprobar préstamos bancarios

## Pregunta

> Un modelo de IA para aprobar préstamos bancarios rechaza sistemáticamente a personas de cierta región geográfica, aunque tienen buen historial crediticio. ¿Qué está ocurriendo? ¿Cómo lo detectarías y corregirías?

## ¿Qué está ocurriendo?

Lo más probable es que el modelo esté mostrando **sesgo algorítmico**. Esto ocurre cuando un sistema de IA produce resultados desfavorables para ciertos grupos de personas, incluso si esos grupos cumplen con los criterios objetivos del proceso.

En este caso, el modelo rechaza a personas de una región geográfica específica aunque tengan buen historial crediticio. Eso sugiere que la región, o variables relacionadas con la región, están influyendo indebidamente en la decisión.

## Posibles causas

### 1. Sesgo histórico en los datos

Si en el pasado se aprobaron menos préstamos a personas de esa región, el modelo puede aprender ese patrón como si fuera una señal válida de riesgo.

Esto es peligroso porque el modelo no distingue entre:

- una relación legítima con el riesgo crediticio;
- una discriminación histórica;
- una mala calidad de datos;
- decisiones humanas previas injustas.

### 2. Variables proxy

Aunque no se use explícitamente la variable “región”, otras variables pueden funcionar como sustitutos indirectos.

Ejemplos:

- código postal;
- ciudad;
- sucursal bancaria;
- nivel promedio de ingresos de la zona;
- tipo de vivienda;
- historial laboral asociado a una zona.

Estas variables pueden permitir que el modelo infiera la región y reproduzca el sesgo.

### 3. Datos desbalanceados

Puede haber pocos ejemplos positivos de esa región en el conjunto de entrenamiento. Si el modelo ve pocos casos de personas de esa zona que hayan recibido préstamos y pagado correctamente, tendrá una visión incompleta del riesgo real.

### 4. Mala calibración del modelo

El modelo puede estar asignando probabilidades de riesgo más altas a ese grupo aunque, en la práctica, el riesgo real sea similar al de otros grupos.

## ¿Cómo detectarlo?

No basta con mirar la precisión global del modelo. Un modelo puede tener buen desempeño general y aun así discriminar contra un subgrupo.

### Evaluación por subgrupos

Separar los resultados por región y comparar métricas como:

- tasa de aprobación;
- tasa de rechazo;
- falsos positivos;
- falsos negativos;
- precisión;
- recall;
- tasa de default real.

Si personas con perfiles similares reciben decisiones distintas dependiendo de la región, hay una señal clara de sesgo.

### Pruebas contrafactuales

Consisten en tomar un mismo perfil de cliente y cambiar solamente la región.

Ejemplo:

| Historial crediticio | Ingresos | Deuda | Región | Decisión |
|---|---:|---:|---|---|
| Bueno | 5,000 | Baja | A | Aprobado |
| Bueno | 5,000 | Baja | B | Rechazado |

Si el único cambio es la región y la decisión cambia, el modelo probablemente está usando la región de forma injusta.

### Explicabilidad del modelo

Se pueden usar técnicas como:

- SHAP;
- LIME;
- análisis de importancia de variables;
- análisis de correlaciones;
- revisión de reglas de decisión.

El objetivo es entender qué variables influyen más en la decisión y si alguna variable sensible o proxy está teniendo demasiado peso.

### Métricas de equidad

Algunas métricas útiles son:

- **Paridad demográfica**: compara si las tasas de aprobación son similares entre grupos.
- **Equal opportunity**: compara si las personas calificadas tienen la misma probabilidad de aprobación.
- **Equalized odds**: compara errores del modelo entre grupos.
- **Calibration by group**: verifica si las probabilidades predichas significan lo mismo para cada grupo.

## ¿Cómo corregirlo?

### 1. Corregir los datos

- Balancear el dataset.
- Agregar datos representativos de la región afectada.
- Revisar etiquetas históricas sesgadas.
- Eliminar variables claramente discriminatorias.
- Detectar y controlar variables proxy.

### 2. Ajustar el modelo

- Entrenar con restricciones de equidad.
- Penalizar diferencias injustificadas entre grupos.
- Usar modelos más interpretables si el caso es regulado.
- Calibrar probabilidades por subgrupo.

### 3. Ajustar el proceso de decisión

- Incorporar revisión humana en decisiones sensibles.
- Crear mecanismos de apelación.
- Documentar por qué se aprueba o rechaza una solicitud.
- Monitorear resultados después del despliegue.

### 4. Auditoría continua

El sesgo puede aparecer después de desplegar el modelo si cambian los datos, la economía o los patrones de solicitud. Por eso se deben monitorear métricas de equidad de forma periódica.

## Conclusión

Este caso muestra que la IA puede amplificar desigualdades si se entrena con datos históricos sesgados. La solución no es solo técnica: requiere gobernanza, auditoría, transparencia y responsabilidad humana.


In [ ]:
# Ejemplo simple: comparación de tasas de aprobación por región

import pandas as pd

data = {
    "region": ["A", "A", "A", "B", "B", "B"],
    "historial_crediticio": ["bueno", "bueno", "regular", "bueno", "bueno", "regular"],
    "aprobado": [1, 1, 0, 0, 0, 0]
}

df = pd.DataFrame(data)

tasas = df.groupby("region")["aprobado"].mean().reset_index()
tasas.rename(columns={"aprobado": "tasa_aprobacion"}, inplace=True)
tasas


---
# 2. Los 6 pilares de la IA Responsable

## Pregunta

> Los 6 Pilares de la IA Responsable

Los pilares de IA Responsable son principios que ayudan a diseñar, evaluar y gobernar sistemas de IA para que sean seguros, justos y confiables.

Aunque distintas organizaciones usan nombres ligeramente diferentes, una formulación común incluye los siguientes seis pilares:

## 1. Equidad

La IA debe tratar a las personas de manera justa y evitar discriminación contra grupos protegidos o vulnerables.

En el caso del modelo de préstamos, este pilar exige revisar si personas de distintas regiones reciben decisiones comparables cuando tienen perfiles crediticios equivalentes.

### Preguntas clave

- ¿El modelo perjudica más a ciertos grupos?
- ¿Existen variables proxy?
- ¿Las tasas de error son similares entre grupos?
- ¿El modelo reproduce sesgos históricos?

## 2. Confiabilidad y seguridad

El sistema debe funcionar de manera consistente, robusta y segura en condiciones normales y adversas.

Esto incluye:

- pruebas antes de producción;
- monitoreo después del despliegue;
- validación con datos nuevos;
- resistencia a entradas maliciosas;
- planes de contingencia.

Un modelo confiable no solo debe ser preciso, sino también estable y controlable.

## 3. Privacidad y seguridad

La IA debe proteger los datos personales y sensibles.

Esto implica:

- minimizar datos;
- cifrar información;
- anonimizar o pseudonimizar;
- controlar accesos;
- evitar fugas en prompts, logs y respuestas;
- cumplir normativas aplicables.

Este pilar es esencial cuando se procesan nombres, SSN, emails, información médica o datos financieros.

## 4. Inclusión

La IA debe estar diseñada para funcionar para una diversidad amplia de personas, contextos y necesidades.

Esto implica considerar:

- idioma;
- accesibilidad;
- cultura;
- ubicación;
- capacidades distintas;
- brechas digitales.

Un sistema puede ser técnicamente correcto pero excluir usuarios si no considera sus condiciones reales.

## 5. Transparencia

Las personas deben poder entender, al menos en términos razonables:

- cuándo están interactuando con IA;
- qué datos se usan;
- qué limitaciones tiene el sistema;
- por qué se tomó una decisión;
- cómo pueden apelar o pedir revisión.

En modelos de alto impacto, como crédito, salud o empleo, la explicabilidad es especialmente importante.

## 6. Responsabilidad

Debe haber responsables humanos por el diseño, despliegue y consecuencias del sistema.

La responsabilidad incluye:

- roles claros;
- auditorías;
- documentación;
- trazabilidad;
- comités de revisión;
- mecanismos de corrección;
- monitoreo continuo.

La IA no debe convertirse en una “caja negra” que elimina la responsabilidad humana.

## Relación entre pilares

Los pilares están conectados. Por ejemplo:

- para lograr equidad, necesitas transparencia;
- para garantizar privacidad, necesitas seguridad;
- para tener responsabilidad, necesitas auditoría;
- para tener confiabilidad, necesitas monitoreo.

También puede haber tensiones. Por ejemplo, explicar más el modelo puede entrar en conflicto con privacidad o seguridad. La gestión responsable consiste en equilibrar estos principios según el riesgo del caso.


---
# 3. Caso: LLM que genera contenido ofensivo o inventado

## Pregunta

> Tu empresa usa un LLM para generar respuestas a clientes. El modelo a veces genera contenido ofensivo o inventado (alucinaciones). ¿Qué mecanismos implementarías para prevenirlo?

## Problema

Los LLMs pueden fallar de dos formas importantes en atención al cliente:

1. **Contenido ofensivo o inapropiado**: respuestas tóxicas, discriminatorias, agresivas o no alineadas con la marca.
2. **Alucinaciones**: respuestas que parecen correctas pero contienen información falsa, inventada o no verificable.

La solución no es confiar únicamente en el modelo. Se debe construir una arquitectura con varias capas de control.

## 1. Controles antes del modelo

Antes de enviar la consulta al LLM, se debe analizar el input.

### Filtros de entrada

Detectan:

- lenguaje ofensivo;
- solicitudes peligrosas;
- intentos de jailbreak;
- instrucciones para ignorar políticas;
- contenido sensible.

### Clasificación de intención

Permite identificar si el usuario pregunta sobre temas de alto riesgo, como:

- salud;
- finanzas;
- asuntos legales;
- quejas graves;
- datos personales;
- cancelaciones o reembolsos.

Las consultas de alto riesgo pueden ir a un flujo especial o a revisión humana.

## 2. Grounding para reducir alucinaciones

Una de las mejores formas de reducir alucinaciones es hacer que el modelo responda con base en fuentes confiables.

### RAG: Retrieval-Augmented Generation

El flujo típico es:

1. El usuario hace una pregunta.
2. El sistema busca información en una base de conocimiento aprobada.
3. El LLM recibe solo los fragmentos relevantes.
4. El LLM responde usando esos fragmentos.
5. La respuesta incluye citas o referencias internas.

### Regla importante

El sistema debe instruir al modelo:

> Si la respuesta no está en las fuentes proporcionadas, indica que no tienes suficiente información.

Esto evita que el modelo “rellene” con información inventada.

## 3. Prompting seguro

El prompt del sistema debe incluir reglas claras:

- no inventar políticas;
- no prometer acciones no disponibles;
- no dar asesoría especializada si no corresponde;
- mantener tono profesional;
- rechazar contenido abusivo;
- escalar casos sensibles;
- responder solo con base en fuentes aprobadas.

## 4. Parámetros de generación

Para atención al cliente, normalmente conviene reducir creatividad.

Configuraciones comunes:

- temperatura baja;
- límite de tokens;
- respuestas estructuradas;
- formato controlado;
- uso de herramientas para datos dinámicos.

Si el cliente pregunta por el estado de una orden, el modelo no debería inventarlo: debería consultar una herramienta o API.

## 5. Moderación de salida

Después de generar la respuesta, se debe revisar antes de mostrarla al usuario.

### Controles de salida

- detección de toxicidad;
- detección de lenguaje ofensivo;
- verificación de cumplimiento de políticas;
- detección de datos personales;
- revisión de factualidad;
- validación contra fuentes.

Si la respuesta falla, se puede:

- bloquear;
- regenerar;
- reformular;
- escalar a humano.

## 6. Human-in-the-loop

Los humanos deben revisar casos donde el riesgo sea alto.

Ejemplos:

- reclamos legales;
- amenazas;
- información médica;
- decisiones financieras;
- clientes molestos;
- respuestas con baja confianza;
- falta de evidencia en la base de conocimiento.

## 7. Monitoreo y evaluación continua

Se deben medir métricas como:

- tasa de alucinación;
- tasa de toxicidad;
- tasa de escalamiento;
- satisfacción del cliente;
- tiempo de resolución;
- respuestas bloqueadas;
- cumplimiento de políticas.

También se deben realizar pruebas de red teaming para encontrar fallos antes de que los clientes los encuentren.

## 8. Fallbacks seguros

Cuando el sistema no sabe, debe decirlo.

Un fallback seguro puede ser:

> No tengo suficiente información para responder con precisión. Te puedo conectar con un agente o revisar la documentación disponible.

Esto es mejor que entregar una respuesta falsa con alta confianza.

## Conclusión

La prevención de alucinaciones y contenido ofensivo requiere una arquitectura de defensa en profundidad: filtros de entrada, RAG, prompts seguros, moderación de salida, herramientas, monitoreo y revisión humana.


In [ ]:
# Ejemplo conceptual: función simple para validar una respuesta antes de mostrarla

def validar_respuesta(respuesta, fuentes_disponibles=True, contiene_toxicidad=False):
    if contiene_toxicidad:
        return "Bloquear o regenerar respuesta"

    if not fuentes_disponibles:
        return "Usar fallback seguro o escalar a humano"

    if "no estoy seguro" in respuesta.lower():
        return "Escalar o pedir más contexto"

    return "Respuesta aprobada"

validar_respuesta(
    respuesta="Según la política disponible, puedes solicitar soporte por el portal.",
    fuentes_disponibles=True,
    contiene_toxicidad=False
)


---
# 4. Caso: Amazon Bedrock y documentos con datos personales

## Pregunta

> Estás usando Amazon Bedrock para procesar documentos de clientes que contienen datos personales (nombres, SSN, emails). ¿Cómo garantizas la privacidad de estos datos?

## Riesgo principal

El riesgo es exponer información personal identificable, también conocida como PII, durante el procesamiento con IA.

Ejemplos de PII:

- nombres;
- direcciones;
- correos electrónicos;
- números de seguro social;
- teléfonos;
- fechas de nacimiento;
- números de cuenta;
- identificadores médicos o financieros.

La estrategia correcta es aplicar privacidad por diseño.

## 1. Minimización de datos

Antes de enviar información a un modelo, se debe preguntar:

> ¿El modelo realmente necesita ver estos datos?

Si no los necesita, deben eliminarse, ocultarse o reemplazarse.

Ejemplos:

- `123-45-6789` → `XXX-XX-6789`
- `ana.perez@email.com` → `[EMAIL_1]`
- `Ana Pérez` → `[CLIENTE_1]`

## 2. Redacción y anonimización antes de Bedrock

Lo ideal es que el LLM no reciba PII en claro.

Pipeline recomendado:

1. Ingesta del documento.
2. Extracción de texto.
3. Detección de PII.
4. Redacción o tokenización.
5. Procesamiento con Bedrock.
6. Rehidratación controlada solo si es estrictamente necesario.

## 3. Servicios útiles de AWS

### Amazon Comprehend

Puede detectar entidades de PII en texto, como nombres, correos y números identificadores.

### Amazon Macie

Ayuda a descubrir y clasificar datos sensibles almacenados en Amazon S3.

### Amazon Textract

Extrae texto de documentos escaneados, PDFs e imágenes. Después de extraer el texto, se puede pasar por detección de PII.

### AWS KMS

Permite cifrar datos con claves administradas y controladas.

### IAM

Controla quién puede acceder a documentos, modelos, buckets y logs.

### CloudTrail

Registra acciones para auditoría.

## 4. Seguridad de red

Para proteger el tráfico:

- usar TLS;
- usar VPC endpoints o AWS PrivateLink cuando aplique;
- evitar exposición por internet público;
- restringir acceso por políticas de red.

## 5. Logging seguro

No se deben guardar prompts y respuestas con PII sin control.

Buenas prácticas:

- desactivar logs innecesarios;
- enmascarar PII antes de guardar;
- cifrar logs;
- limitar retención;
- restringir acceso;
- auditar consultas.

## 6. Controles en salidas

Aunque se anonimice la entrada, la salida también debe revisarse.

Se deben detectar:

- PII filtrada;
- datos reconstruidos;
- contenido sensible innecesario;
- errores de asociación entre tokens y personas.

## 7. Gobernanza

Se deben definir:

- políticas de retención;
- responsables de datos;
- clasificación de información;
- procedimientos de borrado;
- revisión de accesos;
- evaluación de riesgos;
- documentación del flujo de datos.

## Arquitectura recomendada

```text
Documento del cliente
        ↓
Extracción de texto
        ↓
Detección de PII
        ↓
Anonimización / tokenización
        ↓
Procesamiento en Bedrock
        ↓
Validación de salida
        ↓
Respuesta segura
```

## Conclusión

La privacidad no se garantiza únicamente configurando Bedrock. Se garantiza diseñando un flujo completo donde los datos sensibles se minimizan, protegen, auditan y eliminan de acuerdo con políticas claras.


In [ ]:
# Ejemplo simple de enmascaramiento de PII con expresiones regulares

import re

texto = "Cliente Ana Pérez, email ana.perez@example.com, SSN 123-45-6789."

texto = re.sub(r"\b\d{3}-\d{2}-\d{4}\b", "[SSN_REDACTED]", texto)
texto = re.sub(r"[\w\.-]+@[\w\.-]+", "[EMAIL_REDACTED]", texto)

texto


---
# 5. IA en AWS con cumplimiento GDPR e HIPAA

## Pregunta

> Tu organización quiere usar IA pero necesita cumplir con regulaciones de datos (GDPR, HIPAA). ¿Qué servicios y prácticas de AWS te ayudan a cumplir?

## Idea principal

AWS ofrece servicios que ayudan a cumplir requisitos de seguridad, privacidad, auditoría y gobernanza. Sin embargo, el cumplimiento no es automático. La organización sigue siendo responsable de configurar correctamente los servicios y operar el sistema de forma adecuada.

Esto se conoce como modelo de responsabilidad compartida.

## 1. Protección de datos

### Servicios

- **AWS KMS**: gestión de claves de cifrado.
- **Amazon S3 con cifrado SSE-KMS**: almacenamiento seguro.
- **AWS Secrets Manager**: protección de secretos y credenciales.

### Prácticas

- cifrado en tránsito;
- cifrado en reposo;
- rotación de claves;
- control estricto de secretos;
- separación de datos sensibles.

## 2. Clasificación y detección de datos sensibles

### Servicios

- **Amazon Macie**: identifica datos sensibles en S3.
- **Amazon Comprehend**: detecta PII en texto.
- **Amazon Textract**: extrae texto de documentos para posterior clasificación.

### Prácticas

- data minimization;
- anonimización;
- pseudonimización;
- clasificación por sensibilidad;
- inventario de datos.

## 3. Control de acceso

### Servicios

- **AWS IAM**: permisos granulares.
- **AWS Organizations**: administración multi-cuenta.
- **Service Control Policies**: restricciones a nivel organizacional.
- **Amazon Cognito**: identidad de usuarios finales.

### Prácticas

- mínimo privilegio;
- MFA;
- segregación de funciones;
- separación de ambientes;
- revisión periódica de permisos.

## 4. Auditoría y monitoreo

### Servicios

- **AWS CloudTrail**: registro de acciones en la cuenta.
- **Amazon CloudWatch**: métricas y monitoreo.
- **AWS Config**: control de configuraciones.
- **AWS Audit Manager**: recopilación de evidencia de auditoría.
- **Amazon GuardDuty**: detección de amenazas.

### Prácticas

- logging centralizado;
- alertas de accesos sospechosos;
- retención controlada;
- revisión de eventos críticos;
- evidencias para auditoría.

## 5. Gobernanza de datos

### Servicios

- **AWS Lake Formation**: control de acceso sobre data lakes.
- **AWS Glue Data Catalog**: catálogo de datos.
- **Amazon DataZone**: gobierno y descubrimiento de datos.

### Prácticas

- data lineage;
- ownership de datos;
- políticas de retención;
- borrado seguro;
- control de acceso por dominio.

## 6. IA segura y responsable

### Servicios

- **Amazon Bedrock**: uso de modelos fundacionales con controles empresariales.
- **Guardrails for Amazon Bedrock**: restricciones de contenido y temas.
- **Amazon SageMaker**: entrenamiento y despliegue con mayor control.
- **SageMaker Model Monitor**: monitoreo de modelos.
- **SageMaker Clarify**: análisis de sesgo y explicabilidad.

### Prácticas

- no enviar PII innecesaria al modelo;
- usar RAG con fuentes aprobadas;
- validar outputs;
- documentar modelos;
- monitorear drift;
- revisar sesgos.

## 7. GDPR

GDPR se centra en la protección de datos personales de personas en la Unión Europea.

Prácticas relevantes:

- base legal para procesamiento;
- consentimiento cuando aplique;
- minimización de datos;
- derecho al olvido;
- portabilidad;
- limitación de finalidad;
- residencia de datos;
- acuerdos de procesamiento de datos.

## 8. HIPAA

HIPAA aplica a información de salud protegida en Estados Unidos.

Prácticas relevantes:

- firmar un BAA con AWS si aplica;
- usar servicios elegibles para HIPAA;
- cifrar PHI;
- auditar accesos;
- limitar usuarios autorizados;
- mantener registros de actividad;
- definir políticas de retención y respuesta a incidentes.

## Arquitectura de referencia

```text
Ingesta segura
   ↓
Clasificación de datos
   ↓
Redacción / anonimización
   ↓
Almacenamiento cifrado
   ↓
Procesamiento IA
   ↓
Validación de salida
   ↓
Auditoría y monitoreo
   ↓
Retención / borrado controlado
```

## Conclusión

AWS proporciona muchos servicios útiles para cumplir con GDPR e HIPAA, pero el cumplimiento depende de cómo se diseñe, configure y gobierne la solución. La estrategia correcta combina seguridad técnica, privacidad por diseño, auditoría continua y responsabilidad organizacional.


---
# 6. Checklist general para IA Responsable

Usa este checklist para evaluar un sistema de IA antes de llevarlo a producción.

## Datos

- [ ] ¿Los datos contienen PII, PHI u otra información sensible?
- [ ] ¿Se aplica minimización de datos?
- [ ] ¿Se anonimizan o pseudonimizan datos cuando es posible?
- [ ] ¿Los datos están balanceados entre grupos relevantes?
- [ ] ¿Se revisaron sesgos históricos?

## Modelo

- [ ] ¿Se evaluó desempeño global y por subgrupo?
- [ ] ¿Se midieron métricas de equidad?
- [ ] ¿Se documentaron limitaciones?
- [ ] ¿Se validó contra datos recientes?
- [ ] ¿Se monitorea drift?

## Seguridad

- [ ] ¿Los datos están cifrados en tránsito y reposo?
- [ ] ¿Hay control de acceso con mínimo privilegio?
- [ ] ¿Se auditan accesos?
- [ ] ¿Los logs están protegidos?
- [ ] ¿Hay plan de respuesta a incidentes?

## LLMs

- [ ] ¿Hay filtros de entrada?
- [ ] ¿Hay moderación de salida?
- [ ] ¿El modelo usa fuentes confiables?
- [ ] ¿Puede decir “no sé”?
- [ ] ¿Hay escalamiento humano?
- [ ] ¿Se evalúan alucinaciones?

## Gobernanza

- [ ] ¿Hay dueños claros del sistema?
- [ ] ¿Hay auditorías periódicas?
- [ ] ¿Se documentan decisiones?
- [ ] ¿Existen mecanismos de apelación?
- [ ] ¿Se cumplen regulaciones aplicables?


---
# 7. Resumen final

Los casos analizados muestran que la IA Responsable no es solo una cuestión técnica. Requiere combinar:

- diseño ético;
- datos representativos;
- privacidad por diseño;
- seguridad por defecto;
- monitoreo continuo;
- explicabilidad;
- gobernanza;
- responsabilidad humana.

En sistemas de alto impacto, como préstamos, atención al cliente, salud o procesamiento de documentos sensibles, la organización debe diseñar controles desde el inicio y no agregarlos al final.

La regla práctica más importante es:

> No basta con que un sistema de IA funcione; debe funcionar de manera justa, segura, transparente y responsable.
